# Python 文件与 IO 操作完整指南

> 涵盖文本文件读写、CSV/JSON 数据处理、目录与路径管理（pathlib）


## 一、文本文件读写

### 1. 基础读写：`open()` 函数

Python 打开文件的基本方式：

In [ ]:
# ========== 写文件 ==========
with open("hello.txt", "w", encoding="utf-8") as f:
    f.write("你好，Python！\n")
    f.write("第二行内容\n")

# 模式说明：
# "r"  - 只读（默认）
# "w"  - 只写，会覆盖原文件
# "a"  - 追加，在文件末尾添加
# "x"  - 创建新文件，如果已存在会报错
# "r+" - 读写
# "w+" - 读写，会覆盖原文件
# "a+" - 读写，追加模式

# ========== 读文件 ==========
with open("hello.txt", "r", encoding="utf-8") as f:
    content = f.read()          # 读取全部内容
    print(content)

### 2. 逐行读取（推荐大文件用这种方式）

In [ ]:
# 逐行读取
with open("hello.txt", "r", encoding="utf-8") as f:
    for line in f:
        print(line.strip())     # strip() 去掉换行符

# 读取为列表
with open("hello.txt", "r", encoding="utf-8") as f:
    lines = f.readlines()       # 返回列表
    print(f"共 {len(lines)} 行")
    print(lines)

### 3. 为什么用 `with`？

`with` 语句会自动调用 `f.close()`，即使发生异常也会关闭文件。

In [ ]:
# ❌ 不用 with：容易忘记关闭，异常时也不会关闭
f = open("data.txt", "w")
f.write("一些数据")
# f.close()  # 如果上面出错了，这行不会执行！

# ✅ 用 with：自动关闭，即使发生异常
with open("data.txt", "w", encoding="utf-8") as f:
    f.write("一些数据")
# ← 这里自动调用 f.close()

print("文件已安全写入并关闭")

### 4. 大文件处理：逐行读取

对于大文件，不要一次性 `read()` 全部内容，会占用大量内存。

In [ ]:
# 方法1：直接迭代（最推荐，内存友好）
# with open("big_file.txt", "r", encoding="utf-8") as f:
#     for line in f:
#         process(line.strip())

# 方法2：readline() 一次读一行
# with open("big_file.txt", "r", encoding="utf-8") as f:
#     while True:
#         line = f.readline()
#         if not line:
#             break
#         process(line)

# 方法3：分块读取（二进制文件）
# with open("big_file.bin", "rb") as f:
#     while chunk := f.read(4096):  # 每次读 4KB
#         process(chunk)

print("大文件处理策略已展示（注释中）")

## 二、CSV 数据处理

CSV（Comma-Separated Values）是最常见的数据交换格式之一。Python 内置 `csv` 模块处理 CSV 文件。

### 1. 创建示例 CSV 文件

In [ ]:
import csv

# 先创建一个示例 CSV 文件
csv_data = [
    ["name", "age", "city"],
    ["Alice", "25", "Beijing"],
    ["Bob", "30", "Shanghai"],
    ["Charlie", "35", "Guangzhou"]
]

with open("data.csv", "w", encoding="utf-8", newline="") as f:
    writer = csv.writer(f)
    writer.writerows(csv_data)

print("data.csv 已创建")

# 查看内容
with open("data.csv", "r", encoding="utf-8") as f:
    print(f.read())

### 2. 读取 CSV：基础方式

In [ ]:
import csv

with open("data.csv", "r", encoding="utf-8", newline="") as f:
    reader = csv.reader(f)

    # 读取表头
    header = next(reader)
    print(f"表头: {header}")

    # 逐行读取数据
    print("\n数据行:")
    for row in reader:
        print(row)  # 每行是一个列表

### 3. 读取 CSV：DictReader（推荐）

`DictReader` 把每行变成字典，用表头作为键，代码更易读。

In [ ]:
import csv

with open("data.csv", "r", encoding="utf-8", newline="") as f:
    reader = csv.DictReader(f)

    for row in reader:
        print(f"{row['name']} 今年 {row['age']} 岁，住在 {row['city']}")

### 4. 写入 CSV

In [ ]:
import csv

# 方式1：csv.writer
with open("output.csv", "w", encoding="utf-8", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["name", "age", "city"])  # 写入单行
    writer.writerows([
        ["Alice", 25, "Beijing"],
        ["Bob", 30, "Shanghai"]
    ])

# 方式2：DictWriter（推荐）
fieldnames = ["name", "age", "city"]

with open("output2.csv", "w", encoding="utf-8", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()  # 写入表头

    writer.writerow({"name": "Alice", "age": 25, "city": "Beijing"})
    writer.writerows([
        {"name": "Bob", "age": 30, "city": "Shanghai"},
        {"name": "Charlie", "age": 35, "city": "Guangzhou"}
    ])

print("output.csv 和 output2.csv 已创建")

# 验证
with open("output2.csv", "r", encoding="utf-8") as f:
    print(f.read())

### 5. 处理特殊格式

CSV 可以自定义分隔符、引号等。

In [ ]:
import csv

# 自定义分隔符（制表符分隔的 TSV）
tsv_data = "name\tage\tcity\nAlice\t25\tBeijing\n"

with open("data.tsv", "w", encoding="utf-8", newline="") as f:
    f.write(tsv_data)

with open("data.tsv", "r", encoding="utf-8", newline="") as f:
    reader = csv.reader(f, delimiter="\t")
    for row in reader:
        print(row)

# 处理包含逗号的内容（自动加引号）
with open("quoted.csv", "w", encoding="utf-8", newline="") as f:
    writer = csv.writer(f, quoting=csv.QUOTE_MINIMAL)
    writer.writerow(["name", "description"])
    writer.writerow(["Alice", "喜欢 Python, 也喜欢 Go"])  # 自动加引号

with open("quoted.csv", "r", encoding="utf-8") as f:
    print(f.read())

## 三、JSON 数据处理

JSON（JavaScript Object Notation）是 Web 开发中最常用的数据格式，Python 内置 `json` 模块。

### 1. 基础读写

In [ ]:
import json

data = {
    "name": "Alice",
    "age": 25,
    "skills": ["Python", "Go", "Rust"],
    "address": {
        "city": "Beijing",
        "zip": "100000"
    },
    "is_student": False
}

# ========== 写入 JSON ==========
with open("data.json", "w", encoding="utf-8") as f:
    json.dump(data, f, ensure_ascii=False, indent=2)
    # ensure_ascii=False → 中文正常显示
    # indent=2 → 格式化缩进

print("data.json 已写入")

# 查看内容
with open("data.json", "r", encoding="utf-8") as f:
    print(f.read())

### 2. 读取 JSON

In [ ]:
import json

with open("data.json", "r", encoding="utf-8") as f:
    loaded = json.load(f)
    print(f"姓名: {loaded['name']}")
    print(f"技能: {loaded['skills']}")
    print(f"城市: {loaded['address']['city']}")

### 3. 字符串与 JSON 互转

In [ ]:
import json

# Python 对象 → JSON 字符串
data = {"name": "Bob", "age": 30}
json_str = json.dumps(data, ensure_ascii=False, indent=2)
print("JSON 字符串:")
print(json_str)

# JSON 字符串 → Python 对象
parsed = json.loads('{"name": "Alice", "age": 25}')
print(f"\n解析结果: {parsed['name']}")

### 4. 处理非标准类型（如日期）

JSON 标准不支持 datetime 类型，需要自定义编码器。

In [ ]:
import json
from datetime import datetime

class DateEncoder(json.JSONEncoder):
    def default(self, obj):
        if isinstance(obj, datetime):
            return obj.strftime("%Y-%m-%d %H:%M:%S")
        return super().default(obj)

now = datetime.now()
result = json.dumps({"time": now}, cls=DateEncoder)
print(result)

## 四、目录与路径管理：`pathlib`

`pathlib` 是 Python 3.4+ 推荐的路径处理方式，**面向对象**，比 `os.path` 更直观。

核心类：`Path`

### 1. 路径创建与解析

In [ ]:
from pathlib import Path

# ========== 创建路径 ==========
# 当前目录
current = Path(".")
print(f"当前目录绝对路径: {current.resolve()}")

# 拼接路径（用 / 运算符，超直观！）
data_dir = Path("data")
file_path = data_dir / "2024" / "sales.csv"
print(f"拼接路径: {file_path}")

# 从字符串创建
path = Path("/home/user/documents/report.pdf")
print(f"\n文件名: {path.name}")          # report.pdf
print(f"文件名(无后缀): {path.stem}")       # report
print(f"后缀: {path.suffix}")               # .pdf
print(f"父目录: {path.parent}")             # /home/user/documents
print(f"路径各部分: {path.parts}")

### 2. 路径判断与信息

In [ ]:
from pathlib import Path

# 创建测试文件
test_file = Path("test_pathlib.txt")
test_file.write_text("测试内容", encoding="utf-8")

print(f"是否存在: {test_file.exists()}")
print(f"是否是文件: {test_file.is_file()}")
print(f"是否是目录: {test_file.is_dir()}")
print(f"文件大小: {test_file.stat().st_size} 字节")

# 清理
test_file.unlink()
print(f"\n删除后是否存在: {test_file.exists()}")

### 3. 文件操作

In [ ]:
from pathlib import Path
import shutil

file = Path("example.txt")

# 写入文本
file.write_text("Hello World!\n第二行", encoding="utf-8")
print("写入完成")

# 读取文本
content = file.read_text(encoding="utf-8")
print(f"读取内容: \n{content}")

# 追加内容
with file.open("a", encoding="utf-8") as f:
    f.write("第三行（追加）\n")

print(f"追加后内容: \n{file.read_text(encoding='utf-8')}")

# 重命名
file.rename("renamed.txt")
print(f"重命名后: {Path('renamed.txt').exists()}")

# 复制
shutil.copy("renamed.txt", "backup.txt")
print(f"备份文件: {Path('backup.txt').exists()}")

# 清理
Path("renamed.txt").unlink()
Path("backup.txt").unlink()
print("清理完成")

### 4. 目录操作

In [ ]:
from pathlib import Path

# ========== 创建目录 ==========
new_dir = Path("my_folder/sub_folder")
new_dir.mkdir(parents=True, exist_ok=True)
# parents=True   → 自动创建父目录
# exist_ok=True  → 已存在时不报错
print(f"目录创建: {new_dir.exists()}")

# ========== 遍历目录 ==========
# 先创建一些测试文件
(new_dir / "file1.txt").write_text("内容1")
(new_dir / "file2.csv").write_text("内容2")
(new_dir / "sub_folder" / "file3.txt").write_text("内容3", encoding="utf-8")

# 列出当前目录内容
print("\n当前目录内容:")
for item in new_dir.iterdir():
    print(f"  {'[DIR]' if item.is_dir() else '[FILE]'} {item.name}")

# 递归遍历所有 .txt 文件
print("\n所有 txt 文件:")
for file in new_dir.rglob("*.txt"):
    print(f"  {file}")

### 5. 实用路径操作

In [ ]:
from pathlib import Path

# 获取用户主目录
home = Path.home()
print(f"用户主目录: {home}")

# 获取当前工作目录
cwd = Path.cwd()
print(f"当前工作目录: {cwd}")

# 相对路径转绝对路径
rel = Path("data/file.txt")
abs_path = rel.resolve()
print(f"绝对路径: {abs_path}")

# 计算相对路径
path1 = Path("/home/user/project/data")
path2 = Path("/home/user/project/src")
print(f"\n相对路径: {path1.relative_to(path2.parent)}")

# 清理测试目录
import shutil
shutil.rmtree("my_folder", ignore_errors=True)
print("\n测试目录已清理")

## 五、综合实战

### 实战 1：批量处理 CSV 文件

In [ ]:
from pathlib import Path
import csv

def process_all_csv(data_dir: Path):
    """批量处理目录下所有 CSV 文件"""
    results = []

    for csv_file in data_dir.rglob("*.csv"):
        print(f"处理: {csv_file}")

        with csv_file.open("r", encoding="utf-8", newline="") as f:
            reader = csv.DictReader(f)
            for row in reader:
                results.append(row)

    return results

# 创建测试数据
test_dir = Path("batch_data")
test_dir.mkdir(exist_ok=True)

(test_dir / "a.csv").write_text("name,age\nAlice,25\nBob,30", encoding="utf-8")
(test_dir / "sub/b.csv").write_text("name,age\nCharlie,35\n", encoding="utf-8")

# 执行
all_data = process_all_csv(test_dir)
print(f"\n总共读取 {len(all_data)} 条记录")
print(all_data)

# 清理
import shutil
shutil.rmtree(test_dir)

### 实战 2：配置文件管理（JSON）

In [ ]:
from pathlib import Path
import json

class ConfigManager:
    """JSON 配置文件管理器"""

    def __init__(self, config_path: Path):
        self.path = config_path
        self._data = {}
        self.load()

    def load(self):
        if self.path.exists():
            self._data = json.loads(self.path.read_text(encoding="utf-8"))
        else:
            self._data = {}

    def save(self):
        self.path.write_text(
            json.dumps(self._data, ensure_ascii=False, indent=2),
            encoding="utf-8"
        )

    def get(self, key, default=None):
        return self._data.get(key, default)

    def set(self, key, value):
        self._data[key] = value
        self.save()

# 使用
config = ConfigManager(Path("config.json"))
config.set("theme", "dark")
config.set("language", "zh-CN")
print(f"主题: {config.get('theme')}")
print(f"语言: {config.get('language')}")

# 查看文件内容
print("\nconfig.json 内容:")
print(Path("config.json").read_text(encoding="utf-8"))

# 清理
Path("config.json").unlink()

### 实战 3：日志文件归档

In [ ]:
from pathlib import Path
import shutil
from datetime import datetime, timedelta

def archive_old_logs(log_dir: Path, days: int = 7):
    """归档超过指定天数的日志文件"""
    archive_dir = log_dir / "archive"
    archive_dir.mkdir(exist_ok=True)

    cutoff = datetime.now() - timedelta(days=days)
    archived_count = 0

    for log_file in log_dir.glob("*.log"):
        # 获取文件修改时间
        mtime = datetime.fromtimestamp(log_file.stat().st_mtime)

        if mtime < cutoff:
            # 移动到归档目录
            dest = archive_dir / f"{mtime.strftime('%Y%m%d')}_{log_file.name}"
            shutil.move(log_file, dest)
            print(f"归档: {log_file.name} -> {dest.name}")
            archived_count += 1

    return archived_count

# 创建测试日志文件
log_dir = Path("logs")
log_dir.mkdir(exist_ok=True)

(log_dir / "app1.log").write_text("日志内容1")
(log_dir / "app2.log").write_text("日志内容2")

# 模拟旧文件（修改时间为10天前）
old_file = log_dir / "old_app.log"
old_file.write_text("旧日志")
old_time = (datetime.now() - timedelta(days=10)).timestamp()
old_file.touch()  # 先创建
import os
os.utime(old_file, (old_time, old_time))  # 修改时间戳

print(f"归档前文件:")
for f in log_dir.iterdir():
    print(f"  {f.name}")

# 执行归档（7天前的文件）
count = archive_old_logs(log_dir, days=7)
print(f"\n归档了 {count} 个文件")

# 查看结果
print(f"\n归档后:")
for f in log_dir.iterdir():
    if f.is_dir():
        print(f"  [DIR] {f.name}:")
        for af in f.iterdir():
            print(f"    {af.name}")
    else:
        print(f"  [FILE] {f.name}")

# 清理
shutil.rmtree(log_dir)

## 六、知识图谱总结

```
文件与 IO 操作
├── 文本文件
│   ├── open() 模式: r/w/a/x/+
│   ├── with 语句（自动关闭）
│   ├── 编码: utf-8
│   └── 大文件: 逐行迭代
│
├── CSV 处理 (csv 模块)
│   ├── csv.reader / csv.writer
│   ├── csv.DictReader / csv.DictWriter ⭐推荐
│   └── 自定义分隔符、引号
│
├── JSON 处理 (json 模块)
│   ├── json.load / json.dump（文件）
│   ├── json.loads / json.dumps（字符串）
│   └── 自定义编码器 (JSONEncoder)
│
└── 路径管理 (pathlib) ⭐推荐
    ├── Path 对象创建
    ├── 路径运算: / 拼接
    ├── 属性: name, stem, suffix, parent
    ├── 判断: exists(), is_file(), is_dir()
    ├── 遍历: iterdir(), glob(), rglob()
    ├── 文件操作: read_text(), write_text()
    └── 目录操作: mkdir(), rmdir()
```

### 关键要点速查

| 主题 | 核心要点 |
|------|---------|
| **文件读写** | 始终用 `with open(...)`，指定 `encoding="utf-8"` |
| **CSV** | `DictReader`/`DictWriter` 比 `reader`/`writer` 更易读 |
| **JSON** | `ensure_ascii=False` 保证中文正常显示 |
| **pathlib** | 用 `/` 拼接路径，用 `rglob()` 递归查找 |
| **大文件** | 逐行迭代，不要一次性 `read()` |
